In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors
from molpher.core import ExplorationTree as ETree


class HydrolyzeEster(MorphingOperator):
    def __init__(self):
        super(HydrolyzeEster, self).__init__()
        self._name = "Ester Hydrolysis (Generalized)"
        self._matches = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")

    def setOriginal(self, mol):
        super(HydrolyzeEster, self).setOriginal(mol)
        self._matches = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)

        for match in matches:
            carbonyl_c_idx = match[0]   
            carbonyl_o_idx = match[1]  
            ester_o_idx = match[2]      
            alkoxy_c_idx = match[3]   

            carbonyl_c = rdkit_mol.GetAtomWithIdx(carbonyl_c_idx)
            alkoxy_atom = rdkit_mol.GetAtomWithIdx(alkoxy_c_idx)
            
            oxygen_neighbors = [
                nb for nb in carbonyl_c.GetNeighbors()
                if nb.GetAtomicNum() == 8
            ]
            if len(oxygen_neighbors) > 2:
                continue

            # EXCLUSION: tert-butyl esters
            
            carbon_neighbors = [
                nb for nb in alkoxy_atom.GetNeighbors()
                if nb.GetAtomicNum() == 6
            ]
            if len(carbon_neighbors) == 3:
                continue

            self._matches.append(
                (carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx)
            )

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._matches:
            
            return MolpherMol(other=rdkit_mol)

        carbonyl_c_idx, carbonyl_o_idx, ester_o_idx, alkoxy_c_idx = random.choice(self._matches)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            if rw_mol.GetBondBetweenAtoms(carbonyl_c_idx, ester_o_idx):
                rw_mol.RemoveBond(carbonyl_c_idx, ester_o_idx)
            else:
                return MolpherMol(other=rdkit_mol)

            new_oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(carbonyl_c_idx, new_oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            fragments = Chem.GetMolFrags(new_mol, asMols=True, sanitizeFrags=False)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            processed_frags = []
            for frag in fragments:
                frag_rw = Chem.RWMol(frag)
                for atom in frag_rw.GetAtoms():
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

                frag_mol = frag_rw.GetMol()
                try:
                    frag_mol.UpdatePropertyCache(strict=False)
                    Chem.SanitizeMol(frag_mol)
                    processed_frags.append(frag_mol)
                except Exception:
                    continue

            if not processed_frags:
                return MolpherMol(other=rdkit_mol)

            benzene = Chem.MolFromSmarts("c1ccccc1")
            ring_fragments = [f for f in processed_frags if f.HasSubstructMatch(benzene)]

            if ring_fragments:
                largest_frag = max(ring_fragments, key=lambda m: Descriptors.MolWt(m))
            else:
                largest_frag = max(processed_frags, key=lambda m: Descriptors.MolWt(m))

            largest_frag.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(largest_frag, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(largest_frag, cleanIt=True, force=True)

            return MolpherMol(other=largest_frag)

        except Exception as e:
            print(f"[Debug Error]: {e}")
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

hydrolize_ester = HydrolyzeEster()

test_molecules = {
    "1. Ασπιρίνη (Aspirin)": "CC(=O)Oc1ccccc1C(=O)O",
    "2. Μεθυλο-βενζοϊκός εστέρας": "COC(=O)c1ccccc1",
    "3. Οξικός Αιθυλεστέρας (Αλειφατικός)": "CCOC(=O)C",
    "4. Φαινυλο-οξικός εστέρας": "CC(=O)Oc1ccccc1",
    "5. Ηρωίνη (Πολυκυκλικός Διεστέρας)": "CC(=O)OC1C=CC2C3CC4C5=C6C23CCN4CC1OC6=C5OC(=O)C",
    "6. Dimethyl-carbonate": "COC(=O)OC",
    "7.Dibenzyl-carbonate": "O=C(OCC1=CC=CC=C1)OCC2=CC=CC=C2",
    "8. tert-Butyl-ester": "CC(=O)OC(C)(C)C" 
}

print("=== STARTING ESTER HYDROLYSIS TESTING ===")
for name, smiles in test_molecules.items():
    #Έλεγχος εγκυρότητας μέσω RDKit πριν το πάρει το Molpher
    test_mol = Chem.MolFromSmiles(smiles)
    if test_mol is None:
        print(f"\n{name}")
        print(f"  SMILES Error: Το SMILES [{smiles}] δεν είναι έγκυρο.")
        continue
        
    mol = MolpherMol(smiles)
    hydrolize_ester.setOriginal(mol)
    product = hydrolize_ester.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    if product:
        print(f"  TARGET: {product.getSMILES()}")
    else:
        print("  TARGET: Failed (None)")
print("\n=================================")

=== STARTING ESTER HYDROLYSIS TESTING ===

1. Ασπιρίνη (Aspirin)
  SOURCE: CC(=O)OC1=CC=CC=C1C(=O)O
  TARGET: O=C(O)C1=CC=CC=C1O

2. Μεθυλο-βενζοϊκός εστέρας
  SOURCE: COC(=O)C1=CC=CC=C1
  TARGET: O=C(O)C1=CC=CC=C1

3. Οξικός Αιθυλεστέρας (Αλειφατικός)
  SOURCE: CCOC(C)=O
  TARGET: CC(=O)O

4. Φαινυλο-οξικός εστέρας
  SOURCE: CC(=O)OC1=CC=CC=C1
  TARGET: OC1=CC=CC=C1

5. Ηρωίνη (Πολυκυκλικός Διεστέρας)
  SOURCE: CC(=O)OC1=C2OC3CN4CCC56C2=C1C4CC5C6C=CC3OC(C)=O
  TARGET: CC(=O)OC1C=CC2C3CC4C5=C6C(=C5O)OC1CN4CCC623

6. Dimethyl-carbonate
  SOURCE: COC(=O)OC
  TARGET: COC(=O)OC

7.Dibenzyl-carbonate
  SOURCE: O=C(OCC1=CC=CC=C1)OCC1=CC=CC=C1
  TARGET: O=C(OCC1=CC=CC=C1)OCC1=CC=CC=C1

8. tert-Butyl-ester
  SOURCE: CC(=O)OC(C)(C)C
  TARGET: CC(=O)OC(C)(C)C

